# EigenOS Quantum Suite - VQCC v7 submission generator

**One cell. It rebuilds `submission.zip` (18 members, SHA-256
`809a51a9a2f294add361a56430a8c8ed9aefbde4e62190751598abf8f0b5da7c`) from the
public dataset `manwithacat/vqcc-apache-public-circuits`.**

The dataset is optional. If it is attached the cell copies the members out of
`/kaggle/input`; if it is not, the cell downloads the same 18 files from the
public Kaggle dataset API (so turn **Internet** on in the session panel).

Every member is SHA-256 pinned in the cell itself. A missing file, an unpinned
member or a hash mismatch **fails the cell** - it will not package an archive
it cannot vouch for.

The archive is written to `/kaggle/working/submission.zip`. That is the file to
upload to the competition.

Generated by EigenOS Quantum Suite, 5 November 2026. Licence: Apache-2.0.
Provenance for every member: `receipts/v7-submission-manifest.json` in
`EigenOS-Quantum/eigenos-vqcc`.


In [ ]:
import os, sys, json, glob, hashlib, shutil, zipfile, tempfile, subprocess
import urllib.request, urllib.error, urllib.parse, time

DATASET = "manwithacat/vqcc-apache-public-circuits"
DATASET_OWNER, DATASET_SLUG = DATASET.split("/")
DOWNLOAD_BASE = ("https://www.kaggle.com/api/v1/datasets/download/"
                 + DATASET_OWNER + "/" + DATASET_SLUG + "/")

INPUT_ROOT = os.environ.get("VQCC_INPUT_ROOT", "/kaggle/input")
WORK = os.environ.get("VQCC_WORK_DIR", "/kaggle/working")

RUN_OFFICIAL_VERIFIERS = False

LAYOUT = [
    ["invsqrt.kmb", "circuits/invsqrt/af4698478fba.kmb"],
    ["dqi.kmb", "circuits/dqi/993c70386a3d.kmb"],
    ["dfthcsos/submission.npz", "factorizations/b93b4db795e7.npz"],
    ["dfthcsos/angle_k_0.kmb", "circuits/chemistry/a344f422f2e0/angle_k_0.kmb"],
    ["dfthcsos/angle_k_1.kmb", "circuits/chemistry/a344f422f2e0/angle_k_1.kmb"],
    ["dfthcsos/angle_k_2.kmb", "circuits/chemistry/a344f422f2e0/angle_k_2.kmb"],
    ["dfthcsos/angle_k_3.kmb", "circuits/chemistry/a344f422f2e0/angle_k_3.kmb"],
    ["dfthcsos/angle_k_4.kmb", "circuits/chemistry/a344f422f2e0/angle_k_4.kmb"],
    ["dfthcsos/angle_k_5.kmb", "circuits/chemistry/a344f422f2e0/angle_k_5.kmb"],
    ["dfthcsos/angle_k_6.kmb", "circuits/chemistry/a344f422f2e0/angle_k_6.kmb"],
    ["dfthcsos/angle_k_7.kmb", "circuits/chemistry/a344f422f2e0/angle_k_7.kmb"],
    ["dfthcsos/angle_k_8.kmb", "circuits/chemistry/a344f422f2e0/angle_k_8.kmb"],
    ["dfthcsos/angle_k_9.kmb", "circuits/chemistry/a344f422f2e0/angle_k_9.kmb"],
    ["dfthcsos/angle_k_10.kmb", "circuits/chemistry/a344f422f2e0/angle_k_10.kmb"],
    ["dfthcsos/angle_k_11.kmb", "circuits/chemistry/a344f422f2e0/angle_k_11.kmb"],
    ["dfthcsos/angle_k_12.kmb", "circuits/chemistry/a344f422f2e0/angle_k_12.kmb"],
    ["dfthcsos/angle_k_13.kmb", "circuits/chemistry/a344f422f2e0/angle_k_13.kmb"],
    ["dfthcsos/angle_k_14.kmb", "circuits/chemistry/a344f422f2e0/angle_k_14.kmb"],
]
EXPECTED = {
    "invsqrt.kmb": "af4698478fbad775a5a271690cf4c2d8783caf0b8d12048106d1973c4ac6862e",
    "dqi.kmb": "993c70386a3dde6664ecea36315db3aa60fb18a96dc714a18160cae3472378db",
    "dfthcsos/submission.npz": "b463690f734a8c4aae5c19677d7be6ae7589f05d69ad06b93f1c166d671af9df",
    "dfthcsos/angle_k_0.kmb": "2665ce00a1704f63e9d7d3532877568a86bd4d95a90d21e62fd1a9f81ff68298",
    "dfthcsos/angle_k_1.kmb": "4a551d80b9845030ff32fed85066de9d1d2aaf1629d4f560abf4a533f8848034",
    "dfthcsos/angle_k_2.kmb": "67fd7e8f917da9b52ff017787973e447bc3935c8fd768a1c8198d39aa1c6193f",
    "dfthcsos/angle_k_3.kmb": "0d96dcfd083d448eda6ae1156f9037a8f5c044e4db13b8fa4d0b4faf98a6269b",
    "dfthcsos/angle_k_4.kmb": "a2b1bddb016684e770d237d4937abc3773015017a12177cc769217f02d928bb4",
    "dfthcsos/angle_k_5.kmb": "e5a93b28520c35ce8244e2b79fe59ca9f06575ba1890d39788d8d4c5446e049b",
    "dfthcsos/angle_k_6.kmb": "ccdcb391f67e720e53434a1c2deaa6d3499ef86cf8663db42523cca11775638e",
    "dfthcsos/angle_k_7.kmb": "7a4e30f7112cd70d5c1adb1a1b6861eb9ed548c3277ee2e80efb20e49382d5f7",
    "dfthcsos/angle_k_8.kmb": "a04028ecf1002a9f7b43f2b6d4621d76d858264fca008661e22a18900b7268ff",
    "dfthcsos/angle_k_9.kmb": "60c4c540710a1ad71c4e67b48c84383902cd2c11bacf2c9408b44c57a52f0b5e",
    "dfthcsos/angle_k_10.kmb": "e832d29420b4beffc017cc688b0d552354681162faf3cef0ac3b4732d33a4f28",
    "dfthcsos/angle_k_11.kmb": "91d72f75bc5d33feadc33635de06052314ee75c708abcf867e3a3374e8f6029d",
    "dfthcsos/angle_k_12.kmb": "1cc58e1bf53f1020cc46a1720f4dca9db8d82b56024a2ea0119486af6b9add41",
    "dfthcsos/angle_k_13.kmb": "167b7d5a45dcf09139dd31cd8283b8493cde0dbceeba72eea64970398e0d2b78",
    "dfthcsos/angle_k_14.kmb": "929035ff8206ad7879d6255cddf25cd3e46f54a67984aa765a50d84d78550caa",
}
VERIFIED_ARCHIVE_SHA256 = "809a51a9a2f294add361a56430a8c8ed9aefbde4e62190751598abf8f0b5da7c"
VERIFIED_ARCHIVE_BYTES = 537708119
SIZES = {
    "circuits/invsqrt/af4698478fba.kmb": 479055,
    "circuits/dqi/993c70386a3d.kmb": 529710680,
    "factorizations/b93b4db795e7.npz": 1866292,
    "circuits/chemistry/a344f422f2e0/angle_k_0.kmb": 413385,
    "circuits/chemistry/a344f422f2e0/angle_k_1.kmb": 410647,
    "circuits/chemistry/a344f422f2e0/angle_k_2.kmb": 411019,
    "circuits/chemistry/a344f422f2e0/angle_k_3.kmb": 408565,
    "circuits/chemistry/a344f422f2e0/angle_k_4.kmb": 362425,
    "circuits/chemistry/a344f422f2e0/angle_k_5.kmb": 362028,
    "circuits/chemistry/a344f422f2e0/angle_k_6.kmb": 364878,
    "circuits/chemistry/a344f422f2e0/angle_k_7.kmb": 363581,
    "circuits/chemistry/a344f422f2e0/angle_k_8.kmb": 364774,
    "circuits/chemistry/a344f422f2e0/angle_k_9.kmb": 363298,
    "circuits/chemistry/a344f422f2e0/angle_k_10.kmb": 365906,
    "circuits/chemistry/a344f422f2e0/angle_k_11.kmb": 368374,
    "circuits/chemistry/a344f422f2e0/angle_k_12.kmb": 362461,
    "circuits/chemistry/a344f422f2e0/angle_k_13.kmb": 363208,
    "circuits/chemistry/a344f422f2e0/angle_k_14.kmb": 365401,
}

def find_dataset_root():
    slug = DATASET.split("/")[-1]
    cands = []
    for base in (INPUT_ROOT,):
        cands += [os.path.join(base, "datasets", DATASET),
                  os.path.join(base, DATASET),
                  os.path.join(base, slug)]
    for c in cands:
        if os.path.isdir(c) and os.path.exists(os.path.join(c, "circuits")) \
           and os.path.isdir(os.path.join(c, "factorizations")):
            return c
    probe = LAYOUT[0][1]
    probe2 = LAYOUT[2][1]
    for base in (INPUT_ROOT,):
        for hit in sorted(glob.glob(os.path.join(base, "**", probe),
                                 recursive=True)):
            root = hit.split("/" + probe)[0]
            if os.path.exists(os.path.join(root, probe2)):
                return root
    return None


def download_one(rel, dest):
    """Fetch one file from the PUBLIC Kaggle dataset API.

    curl is tried first: urllib silently truncated the 530 MB dqi circuit by
    8.5 MB on the real download, curl pulled the exact 529,710,608 bytes.
    Both paths are size-checked, and the caller hash-checks the result.
    """
    url = DOWNLOAD_BASE + urllib.parse.quote(rel, safe="")
    want_bytes = SIZES.get(rel)
    tmp = dest + ".part"
    last = None
    for attempt in (1, 2, 3, 4):
        try:
            if shutil.which("curl"):
                rc = subprocess.call(
                    ["curl", "-sSL", "--fail", "--retry", "3",
                     "--retry-delay", "3", "--max-time", "1800",
                     "-o", tmp, url])
                if rc != 0:
                    raise IOError("curl exit code %d" % rc)
            else:
                req = urllib.request.Request(
                    url, headers={"User-Agent": "Mozilla/5.0"})
                with urllib.request.urlopen(req, timeout=300) as r, \
                        open(tmp, "wb") as f:
                    shutil.copyfileobj(r, f, 1024 * 1024)
            got_bytes = os.path.getsize(tmp)
            if want_bytes and got_bytes != want_bytes:
                raise IOError("short read: got %d bytes, expected %d"
                              % (got_bytes, want_bytes))
            shutil.move(tmp, dest)
            return
        except Exception as exc:
            last = exc
            if os.path.exists(tmp):
                try:
                    os.remove(tmp)
                except OSError:
                    pass
            time.sleep(4 * attempt)
    raise last


DS = find_dataset_root()
if DS is not None:
    print("dataset root :", DS, "(attached - no download needed)")
else:
    DS = os.path.join(WORK, "vqcc", "source")
    print("dataset root : none attached - downloading the 18 files now.")
    print("              538 MB total, the dqi circuit is 530 MB of it.")
    print("              This takes a few minutes. Let it run.")
    os.makedirs(DS, exist_ok=True)
    for _member, remote in LAYOUT:
        dest = os.path.join(DS, remote)
        if os.path.exists(dest) and os.path.getsize(dest) > 0:
            continue
        os.makedirs(os.path.dirname(dest), exist_ok=True)
        print("   fetching", remote, flush=True)
        try:
            download_one(remote, dest)
        except Exception as exc:
            raise SystemExit(
                "STOP: could not download " + remote + " (" + str(exc) + ").\n"
                "  Fix ONE of these in the Kaggle UI:\n"
                "   (a) right-hand panel -> Session -> turn INTERNET ON, or\n"
                "   (b) right-hand panel -> Add Input -> Datasets -> search\n"
                "       'manwithacat/vqcc-apache-public-circuits' -> Add.\n"
                "  Then press Run All again.")

print("python       :", sys.version.split()[0])

CIRCUITS = os.path.join(WORK, "vqcc", "circuits")
if os.path.isdir(CIRCUITS):
    shutil.rmtree(CIRCUITS)
os.makedirs(CIRCUITS)


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


rows, bad = [], 0
for member, remote in LAYOUT:
    src = os.path.join(DS, remote)
    dst = os.path.join(CIRCUITS, member.replace("/", os.sep))
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    if not os.path.exists(src):
        print("MISSING", remote)
        bad += 1
        continue
    shutil.copyfile(src, dst)
    got = sha256(dst)
    want = EXPECTED.get(member)
    if want is None:
        bad += 1
        os.remove(dst)
        print("%-5s %-30s no expected SHA-256 is pinned" % ("FAIL", member))
        continue
    ok = (got == want)
    if not ok:
        bad += 1
        os.remove(dst)
    rows.append({"member": member, "sha256": got,
                 "bytes": os.path.getsize(dst) if os.path.exists(dst) else 0,
                 "ok": ok, "expected": want, "upstream": remote})
    print("%-5s %-30s %s" % ("OK" if ok else "FAIL", member, got))

print()
print("%d of %d members OK, %d bad" % (len(rows) - bad, len(rows), bad))
if bad:
    raise SystemExit("STOP: a hash did not match. Nothing was packaged.")

OUT = os.path.join(WORK, "submission.zip")
FIXED = (2026, 11, 5, 0, 0, 0)

if os.path.exists(OUT):
    os.remove(OUT)
print("packing (the dqi circuit is 530 MB, so this is the slow step)...")
with zipfile.ZipFile(OUT, "w", zipfile.ZIP_DEFLATED, compresslevel=9) as z:
    for member, _remote in LAYOUT:
        path = os.path.join(CIRCUITS, member.replace("/", os.sep))
        info = zipfile.ZipInfo(member, date_time=FIXED)
        info.compress_type = zipfile.ZIP_DEFLATED
        info.external_attr = 0o644 << 16
        info.create_system = 3
        with open(path, "rb") as fh:
            z.writestr(info, fh.read(), compresslevel=9)

stage = tempfile.mkdtemp(prefix="vqcc_pack_")
with zipfile.ZipFile(OUT) as z:
    names = z.namelist()
    assert len(names) == 18, "expected 18 members, got %d" % len(names)
    z.extractall(stage)

manifest = {"members": []}
for member, _remote in LAYOUT:
    p = os.path.join(stage, member.replace("/", os.sep))
    assert os.path.exists(p), "member missing from the packed archive: " + member
    digest = sha256(p)
    want = EXPECTED[member]
    assert digest == want, "packed member does not match its pin: " + member
    manifest["members"].append({"member": member, "sha256": digest,
                                 "bytes": os.path.getsize(p)})
shutil.rmtree(stage, ignore_errors=True)

manifest["archive_sha256"] = sha256(OUT)
manifest["archive_bytes"] = os.path.getsize(OUT)
os.makedirs(os.path.join(WORK, "receipts"), exist_ok=True)
with open(os.path.join(WORK, "receipts", "submission-manifest.json"), "w") as fh:
    json.dump(manifest, fh, indent=2)

print("archive        :", OUT)
print("archive bytes  :", manifest["archive_bytes"])
print("archive sha256 :", manifest["archive_sha256"])
print("members        :", len(manifest["members"]))
if manifest["archive_sha256"] == VERIFIED_ARCHIVE_SHA256:
    print("archive check  : MATCH - byte-identical to the verified archive")
else:
    print("archive check  : the 18 MEMBERS are pinned and identical; only the")
    print("                 zip compression bytes differ between zlib builds.")
    print("                 Upload this file.")

if RUN_OFFICIAL_VERIFIERS:
    try:
        VERIFIERS = None
        for root in (INPUT_ROOT, "input", "."):
            for hit in sorted(glob.glob(os.path.join(root, "**", "verifiers"),
                                        recursive=True)):
                if all(os.path.isdir(os.path.join(hit, p))
                       for p in ("invsqrt", "dqi", "dfthcsos")):
                    VERIFIERS = hit
                    break
            if VERIFIERS:
                break
        if not VERIFIERS:
            print("verifiers      : not found - skipped")
        else:
            COMP_ROOT = os.path.dirname(os.path.normpath(VERIFIERS))
            env = dict(os.environ)
            env["PYTHONPATH"] = COMP_ROOT
            for pkg, script, args in (
                    ("invsqrt", "verify_submission_invsqrt.py",
                     [os.path.join(CIRCUITS, "invsqrt.kmb")]),
                    ("dqi", "verify_submission_dqi.py",
                     [os.path.join(CIRCUITS, "dqi.kmb")]),
                    ("dfthcsos", "verify.py", [CIRCUITS])):
                path = os.path.join(VERIFIERS, pkg, script)
                if not os.path.exists(path):
                    print("%-10s : verifier not found - skipped" % pkg)
                    continue
                print("=== %s ===" % pkg)
                proc = subprocess.run([sys.executable, path] + args,
                                      capture_output=True, text=True, env=env)
                try:
                    rep = json.loads(proc.stdout)
                    print("   status=%s score=%s" % (rep.get("status"),
                                                     rep.get("score")))
                    with open(os.path.join(WORK, "receipts",
                                           "%s-verify.json" % pkg), "w") as fh:
                        json.dump(rep, fh, indent=2, default=str)
                except Exception:
                    print("   did not return JSON:")
                    print(proc.stdout[-800:])
                    print(proc.stderr[-800:])
    except Exception as exc:
        print("optional verifier step failed (it never affects the archive):",
              exc)
else:
    print("official verifiers: SKIPPED (RUN_OFFICIAL_VERIFIERS = False)")

print()
print("UPLOAD THIS FILE: submission.zip")
print("Do not upload anything else.  Do not upload this notebook.")
